<a href="https://colab.research.google.com/github/alysaqiib/flyrank_internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item for one client (`client_hash_id` × `content_hash_id`).

**Source table:** I will use `fact_content_daily_performance` from the FlyRank warehouse, with `dim_content` when content-level attributes are needed.

**Feature window:** February 2026. These are the signals that would be knowable at the decision moment, the end of February.

**Label window:** March 2026. I will use the following month as the observed outcome window rather than using March information as features.

**Prediction/ranking target:** The goal is to rank content items by refresh opportunity. For this contract, the observed March outcome will be used as the label/proxy for whether a content item needs attention.

**Deliberate exclusion:** I will exclude `trend_direction` and `trend_pct` from the features because they are derived from the same trend information used to define decline and could leak the target.

In [3]:
import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    # Read the Colab Secret if it is not exposed as an environment variable
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

token = get_hf_token()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [token]
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
FACT_MAR = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Label window: March 2026


## Fields: feature / label / context / excluded

### Features

For the first version of the refresh-opportunity score, I will use five features:

1. **GSC impressions** — knowable at the decision moment because February Search Console impressions have already been recorded.
2. **GSC clicks** — knowable at the decision moment because February Search Console clicks have already been recorded.
3. **CTR** — knowable at the decision moment because it can be calculated from February clicks and impressions.
4. **GSC average position** — knowable at the decision moment because February Search Console position data has already been recorded.
5. **GA4 sessions** — knowable at the decision moment because February Analytics session data has already been recorded.

### Label / proxy

The target is an observed future outcome used as a proxy for refresh opportunity. I will use the March 2026 outcome window after using February 2026 as the feature window. This keeps future outcome information separate from the features.

### Context

`client_hash_id` and `content_hash_id` identify the client and content item. They are used for grouping and joining but will not be used as predictive features.

### Excluded

I will exclude `trend_direction` and `trend_pct` because they are derived from trend information and can leak information about the outcome. I will also exclude the client and content IDs from the model features because they are identifiers rather than meaningful predictive signals.

I will also exclude March outcome variables from the February feature set because they would not be available at the decision moment.

## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 — Grain

The expected grain of the daily performance table is one row per client, content item, and report date. I will check for duplicate combinations of these three fields. An empty result means the expected grain holds for the checked partition.
### Query 2 — Row count and date window

I will verify the number of rows in the February 2026 feature partition and confirm its minimum and maximum report dates. This confirms that the selected partition represents the intended feature window.
### Query 3 — Missingness and availability

I will check Search Console availability explicitly using `IS TRUE`. This matters because unavailable data can be represented by zero-filled values, so zero should not automatically be interpreted as real performance.

In [4]:
grain_check = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS n
FROM read_parquet('{FACT_FEB}')
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain combinations:")
grain_check
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{FACT_FEB}')
""").df()

window_check
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS unavailable_rows

FROM read_parquet('{FACT_FEB}')
""").df()

availability_check


Duplicate grain combinations:


,total_rows,available_rows,unavailable_rows
0,7355108,2621783,4733325


### Five initial features

I will use five features for the first refresh-opportunity feature frame. All five are based on information available at the end of the feature window.

1. **Impressions** — knowable at the decision moment because the Search Console impressions for the feature window have already been recorded.
2. **Clicks** — knowable at the decision moment because the Search Console clicks for the feature window have already been recorded.
3. **CTR** — knowable at the decision moment because it can be calculated from clicks and impressions available by the decision moment.
4. **Average position** — knowable at the decision moment because Search Console position data for the feature window has already been recorded.
5. **Content age** — knowable at the decision moment because the content's publication/age information is already available.

I will not add more features at this stage. Keeping the first feature frame to five variables makes it easier to check availability, missingness, and leakage before modeling.

In [5]:
# Inspect the available columns in the February partition

columns_check = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{FACT_FEB}')
""").df()

columns_check[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [6]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_feb,

    SUM(gsc_clicks) AS clicks_feb,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr_feb,

    AVG(gsc_avg_position) AS avg_position_feb,

    SUM(ga4_sessions) AS sessions_feb

FROM read_parquet('{FACT_FEB}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Feature frame shape:", features.shape)

features.head(10)

Feature frame shape: (153559, 7)


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,sessions_feb
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,0.000000,5.500000,NaN
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,20.000000,5.000000,0.0
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,0.000000,5.262333,0.0
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.000000,6.407819,0.0
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,0.000000,6.961538,0.0
5,client_3ffa76342f366962,content_0674cc4ae0f68a90,74.0,1.0,1.351351,8.063910,0.0
6,client_3ffa76342f366962,content_0bbad1e88286bfc6,2.0,0.0,0.000000,4.000000,0.0
7,client_3ffa76342f366962,content_388b1fcec3596848,30.0,3.0,10.000000,5.401852,0.0
8,client_3ffa76342f366962,content_0ffc84b3be2f3bc1,26.0,1.0,3.846154,8.479167,0.0
9,client_3ffa76342f366962,content_456ab2db28595187,100.0,2.0,2.000000,4.330032,0.0


In [7]:
print("Number of features:", len(features.columns) - 2)

print("\nFeature columns:")
print(features.columns.tolist())

print("\nMissing values:")
print(features.isnull().sum())

Number of features: 5

Feature columns:
['client_hash_id', 'content_hash_id', 'impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'sessions_feb']

Missing values:
client_hash_id          0
content_hash_id         0
impressions_feb         0
clicks_feb              0
ctr_feb                 0
avg_position_feb        0
sessions_feb        81138
dtype: int64


### Deliberate leakage experiment

To demonstrate target leakage, I will intentionally add March 2026 clicks to the February feature frame.

March clicks represent future information relative to the February decision moment. Therefore, they would not be available when deciding which content should be prioritized.

This experiment is intentionally invalid. It demonstrates why future outcome information must not be used as a feature.

In [8]:
# Get future March outcome information
march_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_mar
FROM read_parquet('{FACT_MAR}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Intentionally add future information to February features
leaky_features = features.merge(
    march_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

leaky_features.head()

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,sessions_feb,clicks_mar
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,0.0,5.500000,NaN,NaN
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,20.0,5.000000,0.0,0.0
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,0.0,5.262333,0.0,0.0
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.0,6.407819,0.0,0.0
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,0.0,6.961538,0.0,0.0


In [9]:
# Deliberately use the future March clicks as a score
leaky_features["leaky_score"] = leaky_features["clicks_mar"]

print(
    "Correlation between leaked score and March clicks:",
    leaky_features["leaky_score"].corr(leaky_features["clicks_mar"])
)

Correlation between leaked score and March clicks: 1.0


### Leakage removed

The `clicks_mar` feature is removed because it comes from the future outcome window and would not be available at the February decision moment.

The `leaky_score` is also removed because it was created only for the leakage demonstration.

The remaining feature frame contains only information available by the end of February 2026.

In [10]:
# Remove the deliberately leaked information
honest_features = leaky_features.drop(
    columns=["clicks_mar", "leaky_score"]
)

print("Final honest feature columns:")
print(honest_features.columns.tolist())

Final honest feature columns:
['client_hash_id', 'content_hash_id', 'impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'sessions_feb']


## 4. Data limits

## Data limits

A key limitation is uneven data availability across clients. Some clients have limited Search Console or Analytics history, so the February 2026 feature window may not provide equally complete information for every client.

I will use the explicit availability flag rather than treating unavailable data as genuine zero performance. This means the resulting refresh-opportunity ranking may be less reliable for content with limited historical data.

Another limitation is that this first version uses a proxy outcome rather than directly measuring whether a real content refresh caused improvement. Therefore, the score should be treated as decision support for prioritization, not as proof that refreshing a page will improve its performance.

## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.